# 05: Experimento v8: Rótulo limpo (sem cancelamentos avisados no treino)

**O que muda nesta versão:** Igual à v7, mas os no-shows cujo motivo indica cancelamento ou aviso (`falta_avisada`, 5,4% das faltas) saem do treino. O teste mantém o rótulo original, para comparar com a v7.

**Versão anterior:** `05_experimento_v7.ipynb`

## Protocolo (igual em todas as versões)

- **Duas divisões 70/30**, as mesmas em todas as versões, para as comparações serem pareadas:
  - **agrupada:** pacientes sorteados até o teste reunir 30% dos registros; nenhum paciente nos dois lados;
  - **temporal:** os 70% de registros mais antigos para treino, e os 30% mais recentes para teste (corte em set/2019).
    `periodo` fica de fora, porque é constante no treino.
- **Quatro modelos:**
  - Rede Bayesiana reduzida ao Manto de Markov, com critério BIC e com AIC;
  - **regressão logística** (one-hot + padronização; regularização escolhida na validação interna);
  - **LightGBM** (30 tentativas do Optuna na validação interna).
- **Validação interna só com os 70% de treino:** CV agrupada (3 folds) na divisão agrupada; os últimos 25% do
  período de treino na divisão temporal.
- **Incerteza:** intervalos de 95% por **bootstrap reamostrando pacientes** (1.000 reamostragens) e diferenças
  pareadas entre modelos nas mesmas reamostragens.

Código: `src/noshow/registry.py` (entrada `"v8"`), resultados em `resultados/experimentos/v8/`.

> **Histórico exploratório (fases 7 e 8).** Este notebook foi produzido com o protocolo anterior, **antes da
> correção da sobreposição entre treino e teste** (PR #36) e sem a seleção interna de faixas, ESS e balanceamento.
> Os números e conclusões abaixo registram o processo de desenvolvimento, mas **não sustentam os resultados do
> artigo**. O protocolo final está nos notebooks `03_rede_bayesiana`, `04_experimentos` e
> `08_validacao_sensibilidade_e_base_externa`, com resultados em `resultados/artigo_sem_sobreposicao/`.

In [ ]:
import json
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from noshow.benchmark import grouped_bootstrap, paired_difference, to_lgbm
from noshow.registry import EXPERIMENTOS, MODELOS, RESULTADOS, load, load_bases, run, splits

warnings.filterwarnings("ignore")
assert os.environ.get("PYTHONHASHSEED") == "0", "Use o kernel do projeto (PYTHONHASHSEED=0); ver README."

VERSAO = "v8"
ANTERIOR = 'v7'
RECALCULAR = False  # True refaz o experimento (~5 min); False reaproveita resultados/experimentos/v8/

if RECALCULAR or not (RESULTADOS / VERSAO / "predicoes.parquet").exists():  # predições não são versionadas
    M, P = run(VERSAO)
else:
    M, P = load(VERSAO)

BLUE, ORANGE, GRAY, INK, GREEN = "#2a78d6", "#eb6834", "#8a8984", "#52514e", "#1baf7a"
COR = {"RB (BIC)": ORANGE, "RB (AIC)": "#eda100", "Regressão logística": BLUE, "LightGBM": GREEN}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRAY, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
})
print(f"Experimento {VERSAO}: {EXPERIMENTOS[VERSAO]['titulo']}")

## 1. Resultados no teste (30%)

In [ ]:
tabela = M.set_index(["divisao", "modelo"]).loc[[(d, m) for d in ["agrupada", "temporal"] for m in MODELOS]]
tabela[["n_variaveis", "roc_auc", "pr_auc", "brier", "p_media", "auc_interna", "manto"]].style.format(
    {"roc_auc": "{:.4f}", "pr_auc": "{:.4f}", "brier": "{:.4f}", "p_media": "{:.1%}", "auc_interna": "{:.4f}"},
    na_rep="—")

## 2. Intervalos de confiança (bootstrap por paciente)

Uma única divisão 70/30 dá um único número por modelo. O bootstrap estima o quanto esse número variaria com outra
amostra de pacientes: reamostra os pacientes do teste com reposição e recalcula as métricas 1.000 vezes.

In [ ]:
BOOT = {}
for divisao in ["agrupada", "temporal"]:
    sub = P[P.divisao == divisao]
    base_ = sub[sub.modelo == MODELOS[0]].sort_values("linha")
    preds = {m: sub[sub.modelo == m].sort_values("linha").p.to_numpy() for m in MODELOS}
    BOOT[divisao] = grouped_bootstrap(base_.y.to_numpy(), preds, base_.grupo.to_numpy(), n_boot=1000)

ic = pd.DataFrame([
    {"divisao": d, "modelo": m, "métrica": met, "média": BOOT[d][f"{m}|{met}"].mean(),
     "ic_inf": np.quantile(BOOT[d][f"{m}|{met}"], 0.025), "ic_sup": np.quantile(BOOT[d][f"{m}|{met}"], 0.975)}
    for d in BOOT for m in MODELOS for met in ["roc_auc", "pr_auc"]])

fig, axes = plt.subplots(2, 2, figsize=(13, 5.5), sharey=True)
for (i, divisao), (j, met) in [(a, b) for a in enumerate(["agrupada", "temporal"]) for b in enumerate(["roc_auc", "pr_auc"])]:
    ax = axes[i, j]
    for k, m in enumerate(MODELOS[::-1]):
        r = ic[(ic.divisao == divisao) & (ic.modelo == m) & (ic["métrica"] == met)].iloc[0]
        ax.plot([r.ic_inf, r.ic_sup], [k, k], color=COR[m], lw=3, solid_capstyle="round")
        ax.plot(r["média"], k, "o", color=COR[m], ms=8, mec="white", mew=2)
        ax.text(r.ic_sup, k, f"  {r['média']:.3f}", va="center", fontsize=8, color=INK)
    ax.set_yticks(range(len(MODELOS)), MODELOS[::-1])
    ax.set_title(f"{divisao} — {'ROC-AUC' if met == 'roc_auc' else 'PR-AUC'}")
    if met == "roc_auc":
        ax.axvline(0.5, color=GRAY, ls="--", lw=1)
    ax.grid(axis="y", visible=False)
fig.suptitle("Estimativa e IC 95% no teste", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.show()

### Calibração: risco previsto × real

A AUC mede a ordenação; a calibração mede se a probabilidade é confiável como valor. Isso importa quando a
mensageria decide por limiar ("alertar acima de 20% de risco"). Pontos sobre a diagonal = bem calibrado.

In [ ]:
base_cal = load_bases()[EXPERIMENTOS[VERSAO]["base"]]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, divisao in zip(axes, ["agrupada", "temporal"]):
    sub = P[P.divisao == divisao]
    lim = max(sub.p.quantile(0.99), sub.groupby("modelo").y.mean().max()) * 1.4
    ax.plot([0, lim], [0, lim], color=GRAY, ls="--", lw=1)
    for m in MODELOS:
        s_ = sub[sub.modelo == m]
        dec = pd.qcut(s_.p.rank(method="first"), 10, labels=False)
        g = s_.groupby(dec).agg(previsto=("p", "mean"), real=("y", "mean"))
        ax.plot(g.previsto, g.real, "o-", color=COR[m], lw=1.5, ms=5, mec="white", label=m)
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_title(f"{divisao}: decis de risco"); ax.set_xlabel("risco previsto (média do decil)")
axes[0].set_ylabel("taxa real de no-show"); axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

sub = P[P.divisao == "temporal"].assign(
    ano=base_cal.appointment_date.iloc[P[P.divisao == "temporal"].linha].dt.year.to_numpy())
nivel = sub.groupby(["ano", "modelo"]).p.mean().unstack()[MODELOS]
nivel.insert(0, "real", sub[sub.modelo == MODELOS[0]].groupby("ano").y.mean())
print("Divisão temporal: risco médio previsto × taxa real, por ano")
nivel.style.format("{:.1%}")

**Comparação com a Rede Bayesiana (BIC)**, o modelo mais parcimonioso do TCC: diferença pareada de ROC-AUC (outro
modelo − RB BIC), com IC 95%.

In [ ]:
comp = pd.DataFrame([{"divisao": d, "modelo": m, **paired_difference(BOOT[d], m, "RB (BIC)", "roc_auc")}
                     for d in BOOT for m in MODELOS[1:]]).set_index(["divisao", "modelo"])
comp.style.format({"diferença": "{:+.4f}", "ic_inf": "{:+.4f}", "ic_sup": "{:+.4f}", "p": "{:.3f}"}) \
    .map(lambda v: "font-weight: bold" if v < 0.05 else "", subset=["p"])

## 3. Comparação com a versão anterior (v7)

O mesmo modelo na mesma divisão e com **os mesmos registros de teste**: a diferença mede só o efeito da mudança
desta versão.

In [ ]:
_, P_ant = load(ANTERIOR)
linhas = []
for divisao in ["agrupada", "temporal"]:
    for m in MODELOS:
        a = P[(P.divisao == divisao) & (P.modelo == m)].sort_values("linha")
        b = P_ant[(P_ant.divisao == divisao) & (P_ant.modelo == m)].sort_values("linha")
        assert (a.linha.to_numpy() == b.linha.to_numpy()).all()
        boot = grouped_bootstrap(a.y.to_numpy(), {"atual": a.p.to_numpy(), "anterior": b.p.to_numpy()},
                                 a.grupo.to_numpy(), n_boot=1000)
        for met in ["roc_auc", "pr_auc"]:
            linhas.append({"divisao": divisao, "modelo": m, "métrica": met,
                           "anterior": boot[f"anterior|{met}"].mean(), "atual": boot[f"atual|{met}"].mean(),
                           **paired_difference(boot, "atual", "anterior", met)})
evolucao = pd.DataFrame(linhas).set_index(["divisao", "modelo", "métrica"])
evolucao.style.format({"anterior": "{:.4f}", "atual": "{:.4f}", "diferença": "{:+.4f}", "ic_inf": "{:+.4f}",
                       "ic_sup": "{:+.4f}", "p": "{:.3f}"}) \
    .map(lambda v: "font-weight: bold" if v < 0.05 else "", subset=["p"])

## 4. Evolução das versões registradas

Melhor ROC-AUC de cada versão já executada (todas com as mesmas divisões).

In [ ]:
todas = pd.concat([load(v)[0] for v in EXPERIMENTOS if (RESULTADOS / v / "metricas.csv").exists()
                   and list(EXPERIMENTOS).index(v) <= list(EXPERIMENTOS).index(VERSAO)])
evol = todas.pivot_table(index=["divisao", "modelo"], columns="versao", values="roc_auc")
evol.style.format("{:.4f}").highlight_max(axis=1, color="#d6e6f8")

## 5. O que os modelos usam

- **Rede Bayesiana:** o Manto de Markov (coluna `manto` na Seção 1) é o modelo inteiro.
- **LightGBM:** importância média das variáveis pelo **SHAP** (contribuição média absoluta para o log-odds),
  reajustado no treino com os hiperparâmetros escolhidos pelo Optuna.

In [ ]:
import lightgbm as lgb
import shap

cfg = EXPERIMENTOS[VERSAO]
bases = load_bases()
base = bases[cfg["base"]]
parametros = json.loads((RESULTADOS / VERSAO / "parametros.json").read_text())["modelos"]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (divisao, (tr, te)) in zip(axes, splits(bases).items()):
    cat = [c for c in cfg["categoricas"] if not (divisao == "temporal" and c == "periodo")]
    cols = cat + cfg["continuas"]
    Xtr, categorias = to_lgbm(base[cols].iloc[tr], cat)
    hp = {k: v for k, v in parametros[f"{divisao}|LightGBM"].items() if k != "auc_interna"}
    modelo = lgb.LGBMClassifier(**hp, random_state=42, verbose=-1).fit(Xtr, base.no_show.to_numpy()[tr])
    Xte = to_lgbm(base[cols].iloc[te], cat, categorias)[0]
    valores = shap.TreeExplainer(modelo).shap_values(Xte.sample(3000, random_state=0))
    valores = valores[1] if isinstance(valores, list) else valores
    imp = pd.Series(np.abs(valores).mean(axis=0), index=cols).sort_values().tail(12)
    ax.barh(imp.index, imp.values, color=GREEN, height=0.6)
    ax.set_title(f"LightGBM — {divisao}: |SHAP| médio"); ax.grid(axis="y", visible=False)
    ax.tick_params(axis="y", labelsize=8)
plt.tight_layout(); plt.show()

## 6. Registro do experimento

**Data:** 01/11/2026 · **Base de comparação:** v7 · **Mudança:** os 262 no-shows com motivo de cancelamento ou aviso
(5,4% das faltas) saem do treino; o teste mantém o rótulo original

| Divisão temporal | v7 → v8 (ROC-AUC) | Diferença [IC 95%] |
|---|---|---|
| RB (BIC) | 0,539 → 0,544 | +0,006 (n.s., *p* = 0,09) |
| LightGBM | 0,548 → 0,548 | 0,000 (n.s.) |
| Regressão logística | 0,536 → 0,532 | −0,004 (n.s.); PR-AUC −0,004 [−0,007; −0,002] |
| RB (AIC) | 0,525 → 0,506 | **−0,019 [−0,032; −0,007]** |

**Leitura:**
- **O rótulo limpo não trouxe ganho.** Nenhum modelo melhora de forma significativa, e a RB (AIC) piora.
- **O motivo é a escala.** São 262 registros em 4.832 faltas, concentrados em 2017–2018 e 2021. Tirá-los muda pouco o
  que o modelo aprende; para a RB (AIC), que já tinha a estrutura mais instável, a mudança de amostra alterou o Manto.
- **O ruído do rótulo não é o gargalo.** O teste continua medindo no-show como a base o define, e é isso que o
  serviço observa.

**Conclusão:** manter o rótulo original. Registrar no TCC (Seção 2.1.1) que a base mistura no-show e cancelamento
avisado, que o efeito sobre o modelo foi testado e que ele é desprezível.